# Global-Level Depression–Cognition Association Analysis

---

## Purpose

This notebook runs the end-to-end analysis comparing cognitive performance between:
- a **depression cohort** and a **control reference cohort**, and
- **global-level connectivity-derived clusters** within the depression cohort against each other and against controls,

using **robust (median/MAD) z-scoring** and **quantile regression** (R/`quantreg` via `rpy2`).

> **Difference from Module-Level Notebook:** The modular notebook (`09_module_cognitive_associations.ipynb`) works with a pre-built module-level feature matrix that already contains cluster columns. This notebook instead reads a combined cohort CSV for the overall analysis and loads **separate per-modality cluster CSVs** (one per connectivity type) for the per-cluster steps.

---

## High-Level Pipeline

| Step | Label | What happens |
|------|-------|-------------|
| **1** | R environment | Load R packages (`quantreg`), configure rpy2 sinks so R console output goes to a log file |
| **2** | Load & preprocess data | Load the combined cohort CSV via `load_and_rename_cohort_data`; count cohort sizes |
| **3** | Task-wise robust z-scores | Compute median/MAD z-scores (referenced to controls) for every cognitive task; visualize raw and z-scored distributions; run one-sample median tests via quantile regression; apply FDR |
| **4** | Domain composite z-scores | Aggregate task z-scores into 6 cognitive domains (median); visualize; test each domain against zero; apply FDR |
| **5** | Visualizations (overall cohort) | Radar/violin plots of task and domain z-score profiles for the full depression cohort; save z-scored CSV |
| **6** | Per-cluster analysis | For each connectivity modality (`functional`, `structural`, `sfc`): load cluster CSV, repeat Steps 3–5 per cluster, run between-cluster QR + FDR, register significance for radar overlays |

---

## Inputs

### Primary input file
| File | Description |
|------|-------------|
| `data/UKB/cohorts/combined_cohort_<DEPRESSION_CODES>.csv` | Main input table — must contain `eid`, `depression_status` (0=control, 1=depressed), all covariate columns, and all cognitive task columns |

### Per-modality cluster files (Step 6 only — optional)
| File pattern | Description |
|---|---|
| `combined_cohort_<CODES>_global_functional_connectivity_clusters.csv` | Functional connectivity cluster assignments |
| `combined_cohort_<CODES>_global_structural_connectivity_clusters.csv` | Structural connectivity cluster assignments |
| `combined_cohort_<CODES>_global_sfc_connectivity_clusters.csv` | SFC cluster assignments |

Each cluster CSV must contain:
- All columns from the combined cohort
- `Cluster` column: string values `"Cluster 0"` or `"Cluster 1"` for depressed subjects; absent/NaN for controls
- `Connectivity` column: the global connectivity metric used for scatter plots

### Column requirements
- `depression_status`: int 0 (control) or 1 (depressed)
- Covariate columns: `age_at_assessment`, `sex`, `I10`, `Z864`, `F419`

---

## Configurable Constants

| Constant | Default | Description |
|----------|---------|-------------|
| `DEPRESSION` | `["F32"]` | ICD10 code(s) selecting the depression cohort |
| `COVARIATES` | `["age_at_assessment","sex","I10","Z864","F419"]` | Covariates for all quantile regression models |
| `DOMAINS` | 6-domain dict | Mapping from domain name → list of task column names |
| `PLOTS_DIR` | `"reports/plots"` | Output directory for all figures |
| `OUTPUT_DIR` | `"data/UKB/cohorts"` | Output directory for logs and CSV |
| `DATA_DIR` | `"data/UKB/cohorts"` | Input data directory |
| `DATA_FILE` | `"module_connectivity_features_with_covariates.csv"` | Main input CSV file name (must be in `DATA_DIR`) |
| `QUANTILE_REGRESSION_BOOTSTRAP_R` | `1000` | Number of bootstrap resamples for quantile regression |

---

## Outputs

### Figures (`reports/plots/`)
| Pattern | Description |
|---------|-------------|
| `violin_raw_tasks.png` | Raw cognitive task distributions (control vs depression) |
| `violin_task_zscores.png` | Task z-score distributions (depression cohort only) |
| `violin_domain_zscores.png` | Domain composite z-score distributions (depression cohort only) |
| `<CODES>_cognition_task_z_scores.png` | Overall cohort task z-score radar |
| `<CODES>_cognition_domain_z_scores.png` | Overall cohort domain z-score radar |
| `schaefer1000+tian54/<conn>_con/<CODES>_..._task_z_scores_<Cluster_N>.png` | Per-cluster task z-score radars |
| `schaefer1000+tian54/<conn>_con/<CODES>_..._domain_z_scores_<Cluster_N>.png` | Per-cluster domain z-score radars |
| `schaefer1000+tian54/<conn>_con/<CODES>_..._task_z_scores_clusters_violin.png` | Task z-score violin by cluster |
| `schaefer1000+tian54/<conn>_con/<CODES>_..._domain_z_scores_clusters_violin.png` | Domain composite violin by cluster |
| `schaefer1000+tian54/<conn>_con/global_<CODES>_..._task_connectivity.png` | Global connectivity vs task z-scores scatter |
| `schaefer1000+tian54/<conn>_con/global_<CODES>_..._domain_connectivity.png` | Global connectivity vs domain z-scores scatter |

### CSV
| File | Description |
|------|-------------|
| `data/UKB/cohorts/depression_cohort_z_scores_<CODES>.csv` | Depression cohort with all task and domain z-score columns |

### Text logs (`data/UKB/cohorts/`)
| File | Description |
|------|-------------|
| `R_analysis_summary_<CODES>.txt` | R console output and model summaries from every QR call |
| `multiple_testing_summary_<CODES>.txt` | FDR correction tables (raw + adjusted p-values, significance) |
| `robust_z_scores_summary_<CODES>.txt` | Control reference statistics and depression z-score summaries |
| `composite_z_scores_summary_<CODES>.txt` | Domain composite z-score statistics |

---

## Key Helper Functions (from `global_cog_associations_utils`)

| Function | Role |
|----------|------|
| `setup_r_environment()` | Install/load R packages, configure rpy2 |
| `load_and_rename_cohort_data(path)` | Load CSV and canonicalize column names from UKB field IDs |
| `calculate_robust_z_scores(data, vars, …)` | Compute median/MAD referenced z-scores; write log |
| `calculate_composite_z_score(data, z_vars, …)` | Aggregate task z-scores into a domain composite; write log |
| `quantile_regression(tmp_csv, dep_vars, …)` | R `quantreg::rq` wrapper; returns dict of p-values |
| `apply_multiple_testing_correction(p_values, …)` | FDR/Bonferroni correction; write formatted log table |
| `plot_z_scores(data, z_vars, …)` | Radar/violin plots of z-score profiles; saves PNG |
| `plot_cognitive_distributions_violin(data, vars, …)` | Violin distributions per group/cluster; saves PNG |
| `plot_conn_cognition_association(data, …)` | Scatter of global connectivity vs cognitive z-scores; saves PNG |
| `register_radar_overlay_significance(…)` | Cache FDR p-values into registry for multi-cluster radar overlays |

---

## Performance Notes
- Quantile regression with `R=1000` bootstrap resamples is **CPU-intensive**. Steps 3 and 4 each run one QR call; the per-cluster loop (Step 6) runs 4 QR calls per modality × 3 modalities = up to 12 QR calls total.
- To test the pipeline quickly, reduce `R` (e.g. `R=10`) in the quantile regression cells below.
- R and the `quantreg` package must be installed and accessible from your Python environment via `rpy2`.

## Setup: Import Libraries and Configure Paths

In [ ]:
import os
import sys
import pandas as pd

# Add source code directory to path so we can import utility functions
sys.path.insert(0, os.path.abspath('../source_code/clinical_associations'))

from global_cog_associations_utils import (
    setup_r_environment,
    quantile_regression,
    load_and_rename_cohort_data,
    plot_conn_cognition_association,
    apply_multiple_testing_correction,
    calculate_robust_z_scores,
    calculate_composite_z_score,
    plot_z_scores,
    register_radar_overlay_significance,
    plot_cognitive_distributions_violin
)

print("✓ Libraries imported successfully")

## Configuration: Analysis Parameters

In [ ]:
# ---------------------------------------------------------------------------
# Configuration — edit these to change cohort, covariates, or file locations
# ---------------------------------------------------------------------------

ASSOCIATION_with = "cognition"          # analysis target label used in plot filenames
COVARIATES = [
    "age_at_assessment", "sex",         # demographic covariates
    "I10", "Z864", "F419"               # ICD comorbidity / medication covariates
]

DEPRESSION       = ["F32"]              # ICD10 code(s) that define the depression cohort
DEPRESSION_CODES = "_".join(DEPRESSION) # e.g. "F32"; used for all output filenames

PLOTS_DIR  = "../reports/plots"         # output directory for all figures
OUTPUT_DIR = "../data/UKB/cohorts"      # output directory for logs and CSV
DATA_DIR   = "../data/UKB/cohorts"      # location of input CSVs
DATA_FILE  = os.path.join(DATA_DIR, f"combined_cohort_{DEPRESSION_CODES}.csv")

os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
# Pre-create per-modality plot subdirectories
for _conn in ["functional", "structural", "sfc"]:
    os.makedirs(f"{PLOTS_DIR}/schaefer1000+tian54/{_conn}_con", exist_ok=True)

# Log file paths
R_LOG_PATH           = os.path.join(OUTPUT_DIR, f"R_analysis_summary_{DEPRESSION_CODES}.txt")
MT_LOG_PATH          = os.path.join(OUTPUT_DIR, f"multiple_testing_summary_{DEPRESSION_CODES}.txt")
ROBUST_Z_LOG_PATH    = os.path.join(OUTPUT_DIR, f"robust_z_scores_summary_{DEPRESSION_CODES}.txt")
COMPOSITE_Z_LOG_PATH = os.path.join(OUTPUT_DIR, f"composite_z_scores_summary_{DEPRESSION_CODES}.txt")

GROUP_COLUMN = "depression_status"  # 0 = control, 1 = depressed

# Bootstrap iterations for quantile regression (R) standard errors and confidence intervals
QUANTILE_REGRESSION_BOOTSTRAP_R = 5000

# ---------------------------------------------------------------------------
# Cognitive domains — each maps a human-readable label -> list of task columns
# ---------------------------------------------------------------------------
DOMAINS = {
    "processing_speed": [
        "Snap_task_mean_reaction_time",
        "Symbol_digit_substitution_task_correct",
        "Trail_making_A_duration",
        "Trail_making_B_duration",
    ],
    "working_memory": [
        "Reverse_number_recall_task_span",
        "Pairs_matching_task_errors_6_pairs",
    ],
    "executive_function": [
        "Trail_making_B_duration",
        "Tower_rearranging_task_correct",
        "Snap_task_mean_reaction_time",
    ],
    "associative_learning": [
        "Paired_associates_learning_task_correct",
    ],
    "abstract_reasoning": [
        "Matrix_pattern_completion_correct",
        "Fluid_intelligence_score",
    ],
    "verbal_ability": [
        "Vocabulary_score",
    ],
}

# Variables where higher raw score = worse performance → z-scores are reversed
VARS_TO_REVERSE = [
    "Snap_task_mean_reaction_time",
    "Trail_making_A_duration",
    "Trail_making_B_duration",
    "Pairs_matching_task_errors_6_pairs",
]

print("Configuration loaded.")
print(f"  Depression cohort codes : {DEPRESSION_CODES}")
print(f"  Covariates              : {COVARIATES}")
print(f"  Domains                 : {list(DOMAINS.keys())}")
print(f"  Unique tasks            : {len(set(v for vals in DOMAINS.values() for v in vals))}")
print(f"  Data file               : {DATA_FILE}")
print(f"  Plots directory         : {PLOTS_DIR}")

## Step 1: Initialize R Environment

`setup_r_environment()` does three things:

1. **Package check** — calls `install_r_package_if_missing` for `quantreg`. If absent, installs from CRAN via `rpy2`.
2. **rpy2 initialisation** — ensures the R global environment is accessible from Python.
3. **Console sink** — redirects R's console output to `R_LOG_PATH` so all model summaries from subsequent `quantreg::rq` calls are captured in a persistent log.

> **Requirement:** R must be installed and `R_HOME` must be resolvable by `rpy2`.

In [ ]:
print("[1/6] Initializing R environment...")
setup_r_environment()
print("R environment ready. Console output will be appended to:")
print(f"  {R_LOG_PATH}")

## Step 2: Load and Preprocess Data

Load the combined cohort CSV and standardize column names. The `load_and_rename_cohort_data` function:

- Maps UK Biobank field IDs to human-readable variable names
- Ensures required columns are present
- Returns a preprocessed DataFrame ready for analysis

In [ ]:
print("\n[2/6] Loading and preprocessing data...")

# Load cohort CSV and standardize column names
data = load_and_rename_cohort_data(DATA_FILE)

# Count subjects by depression status
n_controls_total = int((data[GROUP_COLUMN] == 0).sum())
n_depressed_total = int((data[GROUP_COLUMN] == 1).sum())

print(f"\nCohort Summary:")
print(f"  Total subjects: {len(data)}")
print(f"  Controls (depression_status=0): {n_controls_total}")
print(f"  Depressed (depression_status=1): {n_depressed_total}")
print(f"\nDataFrame shape: {data.shape}")
print(f"\nFirst few columns: {list(data.columns[:10])}")

# Display a sample of the data
data.head(3)

## Step 3: Calculate Task-Wise Robust Z-Scores

### 3.1 Identify Unique Cognitive Variables

Extract all unique task variables across domains to avoid redundant calculations.

In [ ]:
print("\n[3/6] Calculating task-wise z-scores and testing significance...")

# Get unique cognitive variables across all domains
unique_z_vars = sorted(set(var for domain_vars in DOMAINS.values() for var in domain_vars))

print(f"\nUnique cognitive tasks to analyze: {len(unique_z_vars)}")
for i, var in enumerate(unique_z_vars, 1):
    print(f"  {i}. {var}")

### 3.2 Visualize Raw Task Distributions

Before z-scoring, plot the raw distributions to inspect baseline differences and identify potential data quality issues.

In [ ]:
# Plot raw cognitive task distributions (control vs depression)
plot_cognitive_distributions_violin(
    data=data,
    variables=unique_z_vars,
    group_column=GROUP_COLUMN,
    control_value=0,
    depression_value=1,
    save_path=os.path.join(PLOTS_DIR, "violin_raw_tasks.png"),
    title="Cognitive task distributions (raw): Control vs Depression",
)

print(f"✓ Saved raw task distributions plot to: {PLOTS_DIR}/violin_raw_tasks.png")

### 3.3 Compute Robust Z-Scores

For each cognitive task variable $x$, the z-score is defined relative to the **control group's** median $\tilde{\mu}$ and MAD (median absolute deviation):

$$z_i = \frac{0.6745 \cdot (x_i - \tilde{\mu})}{\text{MAD}}$$

The constant $0.6745$ makes the MAD-based estimator consistent with the standard deviation under normality. Reference statistics are estimated **from controls only**, so $z = 0$ means "equal to the control median". Results are logged to `robust_z_scores_summary_<DEPRESSION_CODES>.txt`.

In [ ]:
# Compute robust z-scores for depression cohort
z_scored_data = calculate_robust_z_scores(
    data,
    vars=unique_z_vars,
    group_column=GROUP_COLUMN,
    control_value=0,
    depression_value=1,
    log_path=ROBUST_Z_LOG_PATH,
    log_context="Overall cohort; connectivity_type: N/A; cluster: N/A"
)

print(f"\n✓ Computed robust z-scores for {len(unique_z_vars)} tasks")
print(f"  Log written to: {ROBUST_Z_LOG_PATH}")
print(f"  New columns added: {[f'{var}_z' for var in unique_z_vars[:3]]}... (and {len(unique_z_vars)-3} more)")

### 3.4 Reverse Z-Scores for Inverse Tasks

For tasks where **higher raw values = worse performance** (e.g., reaction time, errors), multiply z-scores by -1 so that higher z-scores consistently represent better cognitive performance across all tasks.

In [ ]:
# Reverse z-scores for latency/error tasks
# (higher raw value = worse performance → negate so positive z = better)
for var in VARS_TO_REVERSE:
    if f"{var}_z" in z_scored_data.columns:
        z_scored_data[f"{var}_z"] = -z_scored_data[f"{var}_z"]

print(f"Reversed z-scores for: {VARS_TO_REVERSE}")

### 3.5 Visualize Z-Score Distributions

Plot the depression cohort z-score distributions to inspect standardization results.

In [ ]:
# Plot z-score distributions (depression cohort only)
plot_cognitive_distributions_violin(
    data=z_scored_data,
    variables=[f"{var}_z" for var in unique_z_vars],
    group_column=GROUP_COLUMN,
    control_value=0,
    depression_value=1,
    plot_depression_only=True,
    save_path=os.path.join(PLOTS_DIR, "violin_task_zscores.png"),
    title="Cognitive task distributions (z-scores): Depression cohort",
)

print(f"✓ Saved task z-score distributions plot to: {PLOTS_DIR}/violin_task_zscores.png")

### 3.6 Test Task Z-Scores Against Zero (One-Sample)

Run quantile regression to test whether each task z-score differs significantly from zero at the median (τ=0.5):

- **Model**: `task_z ~ age + sex + comorbidities` (intercept test)
- **Null hypothesis**: median z-score = 0
- **Method**: Bootstrap standard errors (R=1000)
- **Output**: p-values for each task

R console output is redirected to `R_analysis_summary_<DEPRESSION_CODES>.txt`.

In [ ]:
# Write temporary CSV for R quantile regression
z_scored_data.to_csv('/tmp/data.csv', index=False)

# Run one-sample quantile regression (test against zero)
p_values_vars_z = quantile_regression(
    tmp_csv_path='/tmp/data.csv',
    dependent_variables=[f'{var}_z' for var in unique_z_vars],
    covariates=COVARIATES,
    group_column=None,  # None = one-sample test
    tau=0.5,  # Test at median
    R=QUANTILE_REGRESSION_BOOTSTRAP_R,   # Bootstrap replicates
    test_against_zero=True,
    return_effects=False,
    r_output_log_path=R_LOG_PATH
)

print(f"\n✓ Completed quantile regression for {len(unique_z_vars)} tasks")
print(f"  R output logged to: {R_LOG_PATH}")
print(f"\nSample p-values (first 3 tasks):")
for var in list(p_values_vars_z.keys())[:3]:
    print(f"  {var}: p = {p_values_vars_z[var]:.4f}")

### 3.7 Apply Multiple Testing Correction (Task-Wise)

Correct for multiple comparisons using FDR (Benjamini-Hochberg) with a false discovery rate of 0.05 across all task-level tests.

Results are logged to `multiple_testing_summary_<DEPRESSION_CODES>.txt`.

In [ ]:
# Apply FDR correction
reject_vars, pvals_vars_corrected = apply_multiple_testing_correction(
    p_values=list(p_values_vars_z.values()),
    variable_names=[f'{var}_z' for var in unique_z_vars],
    test_methods=['Quantile Regression'] * len(unique_z_vars),
    method='fdr_bh',
    alpha=0.05,
    log_path=MT_LOG_PATH,
    log_context="Overall cohort; connectivity_type: N/A; cluster: N/A; level: task-wise"
)

n_significant = reject_vars.sum()
print(f"\n✓ Multiple testing correction complete")
print(f"  Significant tasks (FDR < 0.05): {n_significant} / {len(unique_z_vars)}")
print(f"  Results logged to: {MT_LOG_PATH}")

if n_significant > 0:
    sig_vars = [var for var, sig in zip(unique_z_vars, reject_vars) if sig]
    print(f"\n  Significant tasks:")
    for var in sig_vars:
        print(f"    - {var}")

## Step 4: Calculate Composite Domain Z-Scores

### 4.1 Aggregate Tasks into Domains

Combine task-wise z-scores into composite domain scores using the **median** of constituent tasks.

Results are logged to `composite_z_scores_summary_<DEPRESSION_CODES>.txt`.

In [ ]:
print("\n[4/6] Calculating composite z-scores and testing significance...")

# Calculate composite z-score for each domain
for domain, vars_list in DOMAINS.items():
    z_vars_domain = [f'{var}_z' for var in vars_list]
    z_scored_data = calculate_composite_z_score(
        z_scored_data,
        z_vars=z_vars_domain,
        output_column=f'{domain}',
        method='median',
        log_path=COMPOSITE_Z_LOG_PATH,
        log_context=f"Overall cohort; connectivity_type: N/A; cluster: N/A; domain: {domain}"
    )

print(f"\n✓ Computed composite z-scores for {len(DOMAINS)} domains")
print(f"  Log written to: {COMPOSITE_Z_LOG_PATH}")
print(f"\nDomains:")
for domain in DOMAINS.keys():
    print(f"  - {domain}")

### 4.2 Visualize Domain Z-Score Distributions

In [ ]:
# Plot domain composite z-score distributions
plot_cognitive_distributions_violin(
    data=z_scored_data,
    variables=list(DOMAINS.keys()),
    group_column=GROUP_COLUMN,
    control_value=0,
    depression_value=1,
    plot_depression_only=True,
    save_path=os.path.join(PLOTS_DIR, "violin_domain_zscores.png"),
    title="Cognitive domain distributions (z-scores): Depression cohort",
)

print(f"✓ Saved domain z-score distributions plot to: {PLOTS_DIR}/violin_domain_zscores.png")

### 4.3 Test Domain Z-Scores Against Zero

In [ ]:
# Run one-sample quantile regression for domain composites
z_scored_data.to_csv('/tmp/data.csv', index=False)
p_values_domains_z = quantile_regression(
    tmp_csv_path='/tmp/data.csv',
    dependent_variables=list(DOMAINS.keys()),
    covariates=COVARIATES,
    group_column=None,
    tau=0.5,
    R=QUANTILE_REGRESSION_BOOTSTRAP_R,
    test_against_zero=True,
    return_effects=False,
    r_output_log_path=R_LOG_PATH
)

print(f"\n✓ Completed quantile regression for {len(DOMAINS)} domains")
print(f"\nDomain p-values:")
for domain, pval in p_values_domains_z.items():
    print(f"  {domain}: p = {pval:.4f}")

### 4.4 Apply Multiple Testing Correction (Domain-Wise)

In [ ]:
# Apply FDR correction for domain-level tests
reject_domains, pvals_domains_corrected = apply_multiple_testing_correction(
    p_values=list(p_values_domains_z.values()),
    variable_names=[f'{domain}' for domain in DOMAINS.keys()],
    test_methods=['Quantile Regression'] * len(DOMAINS),
    method='fdr_bh',
    alpha=0.05,
    log_path=MT_LOG_PATH,
    log_context="Overall cohort; connectivity_type: N/A; cluster: N/A; level: domain-wise"
)

n_significant_domains = reject_domains.sum()
print(f"\n✓ Multiple testing correction complete")
print(f"  Significant domains (FDR < 0.05): {n_significant_domains} / {len(DOMAINS)}")

if n_significant_domains > 0:
    sig_domains = [dom for dom, sig in zip(DOMAINS.keys(), reject_domains) if sig]
    print(f"\n  Significant domains:")
    for dom in sig_domains:
        print(f"    - {dom}")

## Step 5: Visualize Overall Cohort Z-Scores

### 5.1 Task-Wise Radar Plot

In [ ]:
print("\n[5/6] Creating z-score visualizations...")

task_z_vars = [f'{var}_z' for var in unique_z_vars]
domain_vars = [f'{domain}' for domain in DOMAINS.keys()]

# Visualize task-wise z-scores (radar/spider plot)
plot_z_scores(
    z_scored_data,
    z_vars=task_z_vars,
    association_type=ASSOCIATION_with,
    type_z_score='task',
    overall_title=(
        'Task Z-Scores (Depression Cohort; '
        f'N_control={n_controls_total}, N_depression={n_depressed_total})'
    ),
    save_path=f'{PLOTS_DIR}/{DEPRESSION_CODES}_{ASSOCIATION_with}_task_z_scores.png',
)

print(f"✓ Saved task z-scores radar plot")

### 5.2 Domain Composite Radar Plot

In [ ]:
# Visualize domain composite z-scores (radar/spider plot)
plot_z_scores(
    z_scored_data,
    z_vars=domain_vars,
    association_type=ASSOCIATION_with,
    type_z_score='domain',
    overall_title=(
        'Domain Composite Z-Scores (Depression Cohort; '
        f'N_control={n_controls_total}, N_depression={n_depressed_total})'
    ),
    save_path=f'{PLOTS_DIR}/{DEPRESSION_CODES}_{ASSOCIATION_with}_domain_z_scores.png',
)

print(f"✓ Saved domain z-scores radar plot")

### 5.3 Save Z-Scored Data

In [ ]:
# Save z-scored depression cohort to CSV
output_file = os.path.join(OUTPUT_DIR, f'depression_cohort_z_scores_{DEPRESSION_CODES}.csv')
z_scored_data.to_csv(output_file, index=False)

print(f"\n✓ Z-scored data saved to: {output_file}")
print(f"  Shape: {z_scored_data.shape}")
print(f"  Columns added: {len([c for c in z_scored_data.columns if c.endswith('_z')])} z-score columns")

## Step 6: Cluster-Specific Cognitive Profiles

For each connectivity modality (functional, structural, SFC), if clustered cohort files exist:

1. Repeat Steps 3–5 for each cluster vs controls
2. Run **between-cluster** quantile regressions
3. Generate cluster-specific visualizations

### 6.1 Process Each Connectivity Modality

In [ ]:
print("\n[6/6] Determining cognitive dysfunction profiles for global connectivity clusters...")

for conn_type in ['functional', 'structural', 'sfc']:
    print(f"\n{'='*80}")
    print(f"Analyzing connectivity type: {conn_type}")
    print(f"{'='*80}")
    
    cluster_csv = os.path.join(DATA_DIR, f"combined_cohort_{DEPRESSION_CODES}_global_{conn_type}_connectivity_clusters.csv")
    
    if not os.path.exists(cluster_csv):
        print(f"  ⚠ No clustered cohort file found at {cluster_csv}")
        print(f"    Skipping {conn_type} connectivity analysis.")
        continue
    
    print(f"\n✓ Detected clustered cohort file: {cluster_csv}")
    cluster_data = load_and_rename_cohort_data(cluster_csv)
    
    # Collect per-cluster z-scored datasets for between-cluster comparisons
    cluster_between_parts = []
    
    # Process each cluster
    for cluster_label in [0, 1]:
        cluster_name = f"Cluster {cluster_label}"
        print(f"\n{'-'*80}")
        print(f"Processing {cluster_name} (depression) vs controls")
        print(f"{'-'*80}")
        
        # Subset: all controls + depressed subjects in current cluster
        cluster_subset = cluster_data[
            (cluster_data[GROUP_COLUMN] == 0) |
            ((cluster_data[GROUP_COLUMN] == 1) & (cluster_data["Cluster"] == cluster_name))
        ].copy()
        
        n_controls_cl = int((cluster_subset[GROUP_COLUMN] == 0).sum())
        n_depressed_cl = int((cluster_subset[GROUP_COLUMN] == 1).sum())
        
        print(f"  N controls: {n_controls_cl}")
        print(f"  N depressed in {cluster_name}: {n_depressed_cl}")
        
        # Step 3: Task-wise robust z-scores
        print(f"\n  Computing task-wise z-scores for {cluster_name}...")
        unique_z_vars_cluster = sorted(set(var for domain_vars in DOMAINS.values() for var in domain_vars))
        z_scored_cluster = calculate_robust_z_scores(
            cluster_subset,
            vars=unique_z_vars_cluster,
            group_column=GROUP_COLUMN,
            control_value=0,
            depression_value=1,
            log_path=ROBUST_Z_LOG_PATH,
            log_context=f"Step 5 within-cluster; connectivity_type: {conn_type}; cluster: {cluster_name}",
        )
        
        # Reverse z-scores for inverse tasks
        for var in [
            "Snap_task_mean_reaction_time",
            "Trail_making_A_duration",
            "Trail_making_B_duration",
            "Pairs_matching_task_errors_6_pairs",
        ]:
            if f"{var}_z" in z_scored_cluster.columns:
                z_scored_cluster[f"{var}_z"] = -z_scored_cluster[f"{var}_z"]
        
        # One-sample quantile regression (tasks)
        print(f"  Running quantile regression (tasks) for {cluster_name}...")
        z_scored_cluster.to_csv("/tmp/data.csv", index=False)
        p_vars_cl = quantile_regression(
            tmp_csv_path="/tmp/data.csv",
            dependent_variables=[f"{var}_z" for var in unique_z_vars_cluster],
            covariates=COVARIATES,
            group_column=None,
            tau=0.5,
            R=QUANTILE_REGRESSION_BOOTSTRAP_R,
            test_against_zero=True,
            return_effects=False,
            r_output_log_path=R_LOG_PATH
        )
        
        apply_multiple_testing_correction(
            p_values=list(p_vars_cl.values()),
            variable_names=[f"{var}_z" for var in unique_z_vars_cluster],
            test_methods=["Quantile Regression"] * len(unique_z_vars_cluster),
            method="fdr_bh",
            alpha=0.05,
            log_path=MT_LOG_PATH,
            log_context=f"Step 5 within-cluster; connectivity_type: {conn_type}; cluster: {cluster_name}; level: task-wise",
        )
        
        # Step 4: Composite domain z-scores
        print(f"  Computing domain composites for {cluster_name}...")
        for domain, vars_list in DOMAINS.items():
            z_vars_dom = [f"{var}_z" for var in vars_list]
            z_scored_cluster = calculate_composite_z_score(
                z_scored_cluster,
                z_vars=z_vars_dom,
                output_column=f"{domain}",
                method="median",
                log_path=COMPOSITE_Z_LOG_PATH,
                log_context=f"Step 5 within-cluster; connectivity_type: {conn_type}; cluster: {cluster_name}; domain: {domain}",
            )
        
        # One-sample quantile regression (domains)
        print(f"  Running quantile regression (domains) for {cluster_name}...")
        z_scored_cluster.to_csv("/tmp/data.csv", index=False)
        p_dom_cl = quantile_regression(
            tmp_csv_path="/tmp/data.csv",
            dependent_variables=list(DOMAINS.keys()),
            covariates=COVARIATES,
            group_column=None,
            tau=0.5,
            R=QUANTILE_REGRESSION_BOOTSTRAP_R,
            test_against_zero=True,
            return_effects=False,
            r_output_log_path=R_LOG_PATH
        )
        
        apply_multiple_testing_correction(
            p_values=list(p_dom_cl.values()),
            variable_names=[f"{domain}" for domain in DOMAINS.keys()],
            test_methods=["Quantile Regression"] * len(DOMAINS),
            method="fdr_bh",
            alpha=0.05,
            log_path=MT_LOG_PATH,
            log_context=f"Step 5 within-cluster; connectivity_type: {conn_type}; cluster: {cluster_name}; level: domain-wise",
        )
        
        # Prepare variables for visualization
        task_z_vars_cl = [f"{var}_z" for var in unique_z_vars_cluster]
        domain_vars_cl = [f"{domain}" for domain in DOMAINS.keys()]
        
        # Save depressed-only rows for between-cluster comparisons
        between_cols = COVARIATES + ["Cluster", "Connectivity_Type", "Connectivity"] + task_z_vars_cl + domain_vars_cl
        between_cols = [c for c in between_cols if c in z_scored_cluster.columns]
        cluster_between_parts.append(
            z_scored_cluster.loc[z_scored_cluster[GROUP_COLUMN] == 1, between_cols].copy()
        )
        
        # Step 5: Visualizations
        print(f"  Creating visualizations for {cluster_name}...")
        
        # Task-wise radar plot
        plot_z_scores(
            z_scored_cluster,
            z_vars=task_z_vars_cl,
            association_type=ASSOCIATION_with,
            type_z_score=f"task_{cluster_name.replace(' ', '_')}",
            overall_title=(
                "Task Z-Scores (Depression Cohort, "
                f"{cluster_name}; N_control={n_controls_cl}, "
                f"N_depression={n_depressed_cl})"
            ),
            save_path=f"{PLOTS_DIR}/schaefer1000+tian54/{conn_type}_con/{DEPRESSION_CODES}_{ASSOCIATION_with}_task_z_scores_{cluster_name.replace(' ', '_')}.png",
        )
        
        # Domain radar plot
        plot_z_scores(
            z_scored_cluster,
            z_vars=domain_vars_cl,
            association_type=ASSOCIATION_with,
            type_z_score=f"domain_{cluster_name.replace(' ', '_')}",
            overall_title=(
                "Domain Composite Z-Scores (Depression Cohort, "
                f"{cluster_name}; N_control={n_controls_cl}, "
                f"N_depression={n_depressed_cl})"
            ),
            save_path=f"{PLOTS_DIR}/schaefer1000+tian54/{conn_type}_con/{DEPRESSION_CODES}_{ASSOCIATION_with}_domain_z_scores_{cluster_name.replace(' ', '_')}.png",
        )
        
        print(f"  ✓ Completed {cluster_name} analysis")
    
    # Between-cluster analyses (if both clusters present)
    if len(cluster_between_parts) >= 2:
        print(f"\n{'-'*80}")
        print(f"Between-Cluster Comparisons ({conn_type})")
        print(f"{'-'*80}")
        
        # Concatenate cluster datasets
        between_df = pd.concat(cluster_between_parts, ignore_index=True)
        between_path = os.path.join(
            "/tmp",
            f"{DEPRESSION_CODES}_{conn_type}_cluster_zscores_concatenated.csv",
        )
        between_df.to_csv(between_path, index=False)
        print(f"  Concatenated dataset: {len(between_df)} subjects")
        print(f"  Saved to: {between_path}")
        
        # Violin plots comparing clusters
        print(f"\n  Creating between-cluster violin plots...")
        
        # Distributions of task z-scores by cluster
        plot_cognitive_distributions_violin(
            data=between_df,
            variables=task_z_vars_cl,
            group_column=GROUP_COLUMN,
            control_value=0,
            depression_value=1,
            plot_depression_clusters=True,
            cluster_column="Cluster",
            cluster_order=("Cluster 0", "Cluster 1"),
            conn_type=conn_type,
            save_path=(
                f"{PLOTS_DIR}/schaefer1000+tian54/{conn_type}_con/{DEPRESSION_CODES}_{ASSOCIATION_with}_task_z_scores_clusters_violin.png"
            ),
            title=(
                f"Task Z-Scores by Cluster ({conn_type.capitalize()} connectivity)"
            ),
        )
        
        # Distribution plots for domain composites by cluster
        plot_cognitive_distributions_violin(
            data=between_df,
            variables=domain_vars_cl,
            group_column=GROUP_COLUMN,
            control_value=0,
            depression_value=1,
            plot_depression_clusters=True,
            cluster_column="Cluster",
            cluster_order=("Cluster 0", "Cluster 1"),
            conn_type=conn_type,
            save_path=(
                f"{PLOTS_DIR}/schaefer1000+tian54/{conn_type}_con/{DEPRESSION_CODES}_{ASSOCIATION_with}_domain_z_scores_clusters_violin.png"
            ),
            title=(
                f"Domain Z-Scores by Cluster ({conn_type.capitalize()} connectivity)"
            ),
        )
        
        # Connectivity–cognition associations (tasks)
        print(f"\n  Creating connectivity-cognition association plots...")
        
        plot_conn_cognition_association(
            data=between_df,
            connectivity_var='Connectivity',
            cognitive_vars=[f'{var}_z' for var in unique_z_vars_cluster],
            group_column='Cluster',
            save_path=f'{PLOTS_DIR}/schaefer1000+tian54/{conn_type}_con/global_{DEPRESSION_CODES}_{ASSOCIATION_with}_task_connectivity.png',
            overall_title=(
                f'Global {conn_type.capitalize()} Connectivity vs Cognitive Z-Scores'
            )
        )
        
        # Between-cluster quantile regression (tasks)
        print(f"\n  Running between-cluster quantile regression (tasks)...")
        
        p_vars_cls = quantile_regression(
            tmp_csv_path=between_path,
            dependent_variables=[f"{var}_z" for var in unique_z_vars_cluster],
            covariates=COVARIATES,
            group_column="Cluster",
            reference_group="Cluster 1",
            comparison_groups=("Cluster 0",),
            test_against_zero=False,
            return_effects=False,
            tau=0.5,
            R=QUANTILE_REGRESSION_BOOTSTRAP_R,
            r_output_log_path=R_LOG_PATH
        )
        
        reject_vars_cls, pvals_vars_cls = apply_multiple_testing_correction(
            p_values=list(p_vars_cls.values()),
            variable_names=[f"{var}_z" for var in unique_z_vars_cluster],
            test_methods=["Quantile Regression"] * len(unique_z_vars_cluster),
            method="fdr_bh",
            alpha=0.05,
            log_path=MT_LOG_PATH,
            log_context=f"Step 5 between-cluster; connectivity_type: {conn_type}; clusters: Cluster 1 (ref) vs Cluster 0; level: task-wise",
        )
        
        register_radar_overlay_significance(
            depression_codes=DEPRESSION_CODES,
            association_type=ASSOCIATION_with,
            base_kind="task",
            conn_type=conn_type,
            variable_names=[f"{var}_z" for var in unique_z_vars_cluster],
            pvals_corrected=pvals_vars_cls,
            comparison_label="Cluster 0 vs Cluster 1 (between-cluster, FDR)",
        )
        
        # Connectivity–cognition associations (domains)
        plot_conn_cognition_association(
            data=between_df,
            connectivity_var='Connectivity',
            cognitive_vars=list(DOMAINS.keys()),
            group_column='Cluster',
            save_path=f'{PLOTS_DIR}/schaefer1000+tian54/{conn_type}_con/global_{DEPRESSION_CODES}_{ASSOCIATION_with}_domain_connectivity.png',
            overall_title=(
                f'Global {conn_type.capitalize()} Connectivity vs Cognitive Z-Scores'
            )
        )
        
        # Between-cluster quantile regression (domains)
        print(f"  Running between-cluster quantile regression (domains)...")
        
        p_dom_cls = quantile_regression(
            tmp_csv_path=between_path,
            dependent_variables=list(DOMAINS.keys()),
            covariates=COVARIATES,
            group_column="Cluster",
            reference_group="Cluster 1",
            comparison_groups=("Cluster 0",),
            test_against_zero=False,
            return_effects=False,
            tau=0.5,
            R=QUANTILE_REGRESSION_BOOTSTRAP_R,
            r_output_log_path=R_LOG_PATH
        )
        
        reject_dom_cls, pvals_dom_cls = apply_multiple_testing_correction(
            p_values=list(p_dom_cls.values()),
            variable_names=[f"{domain}" for domain in DOMAINS.keys()],
            test_methods=["Quantile Regression"] * len(DOMAINS),
            method="fdr_bh",
            alpha=0.05,
            log_path=MT_LOG_PATH,
            log_context=f"Step 5 between-cluster; connectivity_type: {conn_type}; clusters: Cluster 1 (ref) vs Cluster 0; level: domain-wise",
        )
        
        register_radar_overlay_significance(
            depression_codes=DEPRESSION_CODES,
            association_type=ASSOCIATION_with,
            base_kind="domain",
            conn_type=conn_type,
            variable_names=[f"{domain}" for domain in DOMAINS.keys()],
            pvals_corrected=pvals_dom_cls,
            comparison_label="Cluster 0 vs Cluster 1 (between-cluster, FDR)",
        )
        
        print(f"  ✓ Completed between-cluster comparisons for {conn_type}")
    else:
        print(f"\n  ⚠ Skipping between-cluster analysis (missing one or both clusters)")

print(f"\n{'='*80}")
print("Cluster-specific analyses complete!")
print(f"{'='*80}")

## Summary of Outputs

All outputs are written during the analysis. The cell below lists all expected paths and checks whether each file already exists.

In [ ]:
print("=" * 70)
print("Analysis complete — expected output files:")
print("=" * 70)

# ---- Text logs ----
print("\n[Text logs]")
for label, path in [
    ("R console / model summaries   ", R_LOG_PATH),
    ("Multiple testing corrections  ", MT_LOG_PATH),
    ("Robust z-score computation    ", ROBUST_Z_LOG_PATH),
    ("Composite z-score computation ", COMPOSITE_Z_LOG_PATH),
]:
    exists = "✓" if os.path.exists(path) else "✗ (not yet created)"
    print(f"  {label}: {path}  [{exists}]")

# ---- CSV ----
print("\n[CSV]")
output_csv = os.path.join(OUTPUT_DIR, f"depression_cohort_z_scores_{DEPRESSION_CODES}.csv")
exists = "✓" if os.path.exists(output_csv) else "✗"
print(f"  Depression cohort z-scores    : {output_csv}  [{exists}]")

# ---- Overall cohort figures ----
print("\n[Figures — overall cohort]")
for label, path in [
    ("Raw task distributions violin ", os.path.join(PLOTS_DIR, "violin_raw_tasks.png")),
    ("Task z-score violin           ", os.path.join(PLOTS_DIR, "violin_task_zscores.png")),
    ("Domain z-score violin         ", os.path.join(PLOTS_DIR, "violin_domain_zscores.png")),
    ("Task z-score radar            ", f"{PLOTS_DIR}/{DEPRESSION_CODES}_{ASSOCIATION_with}_task_z_scores.png"),
    ("Domain z-score radar          ", f"{PLOTS_DIR}/{DEPRESSION_CODES}_{ASSOCIATION_with}_domain_z_scores.png"),
]:
    exists = "✓" if os.path.exists(path) else "✗"
    print(f"  [{exists}] {path}")

# ---- Per-cluster figures ----
print("\n[Figures — per cluster]")
for conn_type in ["functional", "structural", "sfc"]:
    base = f"{PLOTS_DIR}/schaefer1000+tian54/{conn_type}_con"
    for cluster_label in ["0", "1"]:
        cluster_name = f"Cluster_{cluster_label}"
        for kind in ["task", "domain"]:
            p = f"{base}/{DEPRESSION_CODES}_{ASSOCIATION_with}_{kind}_z_scores_{cluster_name}.png"
            exists = "✓" if os.path.exists(p) else "✗"
            print(f"  [{exists}] {p}")
    for kind in ["task", "domain"]:
        p = f"{base}/{DEPRESSION_CODES}_{ASSOCIATION_with}_{kind}_z_scores_clusters_violin.png"
        exists = "✓" if os.path.exists(p) else "✗"
        print(f"  [{exists}] {p}")
    for kind in ["task", "domain"]:
        p = f"{base}/global_{DEPRESSION_CODES}_{ASSOCIATION_with}_{kind}_connectivity.png"
        exists = "✓" if os.path.exists(p) else "✗"
        print(f"  [{exists}] {p}")

print("\n✓ = file present  |  ✗ = not yet written (run cells above first)")